# 7.03 训练 mini-π0：两阶段训练与评估

[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROI-su7/vla-ladder/blob/main/notebooks/07_vla/7.03_train_mini_pi0.ipynb)


## 为什么要学

π0 的训练分两大阶段：

1. **VLM 预训练**：PaliGemma 在海量图文数据上学会"看图读字"（Google 做好的，openpi 直接加载）；
2. **VLA 训练**：在机器人数据上学动作。π0 先在 1 万小时的多机器人数据上预训练（得到 `pi0_base`），你再在自己的 ALOHA 数据上微调。

本课在玩具桌面上完整走一遍这两个阶段，并做一个关键的消融实验：**如果没有 VLM 预训练会怎样？** 结果会让你直观地理解：为什么 VLA 一定要建立在预训练 VLM 之上。

> ⏱️ 本课在 GPU 上约 6~8 分钟。Colab 请选择 GPU 运行时；CPU 上会非常慢（约 1 小时）。

---

## 1. 数据：示教 → 动作块

In [ ]:
# 🔧 环境准备（每个 notebook 都先运行这一格）
# - 在 Colab 中：自动下载本项目代码
# - 在本地：自动把项目根目录加入 sys.path（也可以在根目录执行 `pip install -e .`）
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/vla-ladder"):
        os.system("git clone -q --depth 1 https://github.com/ROI-su7/vla-ladder.git /content/vla-ladder")
    sys.path.insert(0, "/content/vla-ladder")
else:
    _p = os.path.abspath(".")
    while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "vla_ladder")):
        _p = os.path.dirname(_p)
    sys.path.insert(0, _p)
print("环境就绪 ✅")

In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from vla_ladder.envs import ToyTableEnv, CharTokenizer, collect_table_demos
from vla_ladder.envs.toy_table import COLOR_EN
from vla_ladder.models.mini_pi0 import MiniPi0, MiniPi0Config
from vla_ladder.models.transformer import make_att_2d_masks
from vla_ladder.pipelines import default_cache_dir
from vla_ladder.utils import set_seed, get_device, count_params

device = get_device()
tok = CharTokenizer()
H = 10                                      # 动作块长度（π0 是 50）
env = ToyTableEnv(num_envs=200, seed=0)
t0 = time.time()
raw = collect_table_demos(env, n_episodes=2000, noise=0.1, seed=0)
print(f"采集 2000 条示教，共 {len(raw['state'])} 帧，耗时 {time.time() - t0:.1f}s；专家成功率 {raw['success_rate']:.2f}")

fig, axes = plt.subplots(1, 4, figsize=(11, 3))
for ax, i in zip(axes, [0, 5, 10, 15]):
    ax.imshow(raw["image"][i].transpose(1, 2, 0), extent=(-1, 1, -1, 1)); ax.set_title(f"ep {raw['episode'][i]} t={raw['t'][i]}"); ax.axis("off")
plt.suptitle("frames of one demo (the ring is the gripper)"); plt.show()

### 练习 1：在回合边界内切动作块

数据是所有回合的帧首尾相连拼起来的，`raw["episode"][i]` 是第 i 帧属于哪个回合。动作块**不能跨越回合边界**：越界的部分用本回合最后一帧的动作补齐，并标记为 padding。

In [ ]:
def make_chunk_index(episode, H):
    """
    episode: (F,) 每帧的回合编号（同一回合的帧是连续的）
    返回 idx (F, H)：第 i 帧的动作块用到的帧下标；pad (F, H) bool：True 表示越过了回合末尾
    """
    F_ = len(episode)
    # TODO:
    # 1) last[i] = 第 i 帧所在回合的最后一帧下标（提示：从后往前扫描；或用 np.unique(..., return_index) ）
    # 2) raw = i + arange(H)；idx = minimum(raw, last[:, None])；pad = raw > last[:, None]
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def make_chunk_index(episode, H):
    """
    episode: (F,) 每帧的回合编号（同一回合的帧是连续的）
    返回 idx (F, H)：第 i 帧的动作块用到的帧下标；pad (F, H) bool：True 表示越过了回合末尾
    """
    F_ = len(episode)
    last = np.zeros(F_, dtype=int)
    for i in range(F_ - 1, -1, -1):
        last[i] = i if (i == F_ - 1 or episode[i + 1] != episode[i]) else last[i + 1]
    raw = np.arange(F_)[:, None] + np.arange(H)[None, :]
    return np.minimum(raw, last[:, None]), raw > last[:, None]

In [ ]:
# ✅ 自测
idx, pad = make_chunk_index(np.array([0, 0, 0, 1, 1]), H=3)
assert idx.tolist() == [[0, 1, 2], [1, 2, 2], [2, 2, 2], [3, 4, 4], [4, 4, 4]]
assert pad.tolist()[1] == [False, False, True] and pad.tolist()[3] == [False, False, True]
idx, pad = make_chunk_index(raw["episode"], H)
ids, msk = tok.encode(raw["instruction"])
data = {
    "image": torch.tensor(raw["image"]).to(device),               # uint8，用的时候再 /255（省显存）
    "lang": torch.tensor(ids).to(device), "lang_mask": torch.tensor(msk).to(device),
    "state": torch.tensor(raw["state"]).to(device),
    "actions": torch.tensor(raw["action"])[torch.tensor(idx)].to(device),   # (F, H, 2)
    "action_pad": torch.tensor(pad).to(device),
}
print("通过 ✅", {k: tuple(v.shape) for k, v in data.items()})

> **真实数据的对应**：openpi 用 LeRobot 数据集，`delta_timestamps` 指定"取当前帧之后 50 帧的动作"。数据集已经处理好了回合边界（越界时重复最后一帧，并给出 `action_is_pad`）。

---

## 2. 阶段 1：VLM 预训练（视觉-语言对齐）

我们没有互联网规模的数据，但可以用一个**便宜的代理任务**模拟 VLM 预训练：随机生成 2 万张桌面图像和指令，让 VLM（视觉塔 + 词嵌入 + VLM 层）学会"根据指令在图里找到目标方块"，也就是 4.04 课的 grounding 任务。**这一步不需要任何机器人示教**，就像 PaliGemma 不需要机器人数据一样。

### 练习 2：用 VLM 的文字 token 输出做池化

In [ ]:
def prefix_text_feature(model, images, lang, lang_mask):
    """只跑前缀（VLM 那一套权重），对文字 token 的输出做 masked 平均池化 -> (B, vlm_width)"""
    embs, pad, att = model.embed_prefix(images, lang, lang_mask)
    pos = torch.cumsum(pad.long(), 1) - 1
    (out, _), _ = model.moe([embs, None], make_att_2d_masks(pad, att), pos)
    # TODO: 文字 token 在 out 中的位置是 [n_patches : n_patches + L]；用 lang_mask 做加权平均
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def prefix_text_feature(model, images, lang, lang_mask):
    """只跑前缀（VLM 那一套权重），对文字 token 的输出做 masked 平均池化 -> (B, vlm_width)"""
    embs, pad, att = model.embed_prefix(images, lang, lang_mask)
    pos = torch.cumsum(pad.long(), 1) - 1
    (out, _), _ = model.moe([embs, None], make_att_2d_masks(pad, att), pos)
    n = model.vision.n_patches
    txt = out[:, n : n + lang.shape[1]]
    w = lang_mask.float()[..., None]
    return (txt * w).sum(1) / w.sum(1)

In [ ]:
def pretrain_vlm(model, n_images=20000, steps=4000, B=128, lr=1e-3, seed=100):
    pe = ToyTableEnv(num_envs=n_images, seed=seed); o = pe.reset()
    imgs = torch.tensor(o["image"]).to(device)
    ids_, msk_ = tok.encode(o["instruction"])
    L, M = torch.tensor(ids_).to(device), torch.tensor(msk_).to(device)
    target = torch.tensor(pe.target, dtype=torch.float32).to(device)
    head = nn.Linear(model.cfg.vlm_width, 2).to(device)
    params = [p for m in (model.vision, model.embed_tokens, model.moe.vlm_layers, model.moe.vlm_norm) for p in m.parameters()]
    opt = torch.optim.AdamW(params + list(head.parameters()), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps, pct_start=0.05)
    t0 = time.time()
    for step in range(steps):
        i = torch.randint(0, n_images, (B,), device=device)
        loss = ((head(prefix_text_feature(model, imgs[i], L[i], M[i])) - target[i]) ** 2).mean()
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        if step % 1000 == 0 or step == steps - 1:
            print(f"  [VLM 预训练] step {step:5d}  定位误差² {loss.item():.4f}  ({time.time() - t0:.0f}s)")

set_seed(0)
cfg = MiniPi0Config(vocab_size=tok.vocab_size, action_horizon=H)
model = MiniPi0(cfg).to(device)
print(f"mini-π0 参数量：{count_params(model) / 1e6:.2f} M")
pretrain_vlm(model)

---

## 3. 阶段 2：VLA 训练（流匹配）

`MiniPi0.forward` 和 openpi 一样返回**逐元素**的损失 `(B, H, A)`。我们要把补齐的动作排除在外。

### 练习 3：带 padding 掩码的流匹配损失与训练循环

In [ ]:
def train_vla(model, data, steps=8000, B=128, lr=1e-3):
    opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps, pct_start=0.05)
    N, t0, hist = len(data["state"]), time.time(), []
    model.train()
    for step in range(steps):
        i = torch.randint(0, N, (B,), device=device)
        per_elem = model(data["image"][i].float() / 255, data["lang"][i], data["lang_mask"][i], data["state"][i], data["actions"][i])
        # TODO: valid = (~action_pad[i]) 扩展到 (B, H, 1)；loss = (per_elem * valid).sum() / (valid.sum() * A)
        #       然后 zero_grad / backward / clip_grad_norm_(1.0) / step / sched.step()
        raise NotImplementedError
        hist.append(loss.item())
        if step % 1000 == 0 or step == steps - 1:
            print(f"  [VLA 训练] step {step:5d}  loss {loss.item():.4f}  ({time.time() - t0:.0f}s)")
    model.eval()
    return hist

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def train_vla(model, data, steps=8000, B=128, lr=1e-3):
    opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps, pct_start=0.05)
    N, t0, hist = len(data["state"]), time.time(), []
    model.train()
    for step in range(steps):
        i = torch.randint(0, N, (B,), device=device)
        per_elem = model(data["image"][i].float() / 255, data["lang"][i], data["lang_mask"][i], data["state"][i], data["actions"][i])
        valid = (~data["action_pad"][i]).float()[..., None]
        loss = (per_elem * valid).sum() / (valid.sum() * per_elem.shape[-1])
        opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step()
        hist.append(loss.item())
        if step % 1000 == 0 or step == steps - 1:
            print(f"  [VLA 训练] step {step:5d}  loss {loss.item():.4f}  ({time.time() - t0:.0f}s)")
    model.eval()
    return hist

In [ ]:
hist_pre = train_vla(model, data)

---

## 4. 评估

### 练习 4：闭环评估

每次推理得到 10 步的动作块，执行前 5 步再重新推理（π0 在 ALOHA 上通常推理 50 步、执行 25 步左右）。

In [ ]:
@torch.no_grad()
def evaluate(model, n=200, radius=0.05, exec_steps=5, seed=123, return_env=False):
    e = ToyTableEnv(num_envs=n, seed=seed, success_radius=radius)
    obs = e.reset()
    while not e.done.all():
        ids_, msk_ = tok.encode(obs["instruction"])
        # TODO: chunk = model.sample_actions(图像, 指令, 指令mask, 状态)（都转成 device 上的张量），clamp 到 [-1,1]
        #       执行前 exec_steps 步
        raise NotImplementedError
    res = {"success": float(info["success"].mean()), "median_dist": float(np.median(info["dist"]))}
    return (res, e) if return_env else res

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
@torch.no_grad()
def evaluate(model, n=200, radius=0.05, exec_steps=5, seed=123, return_env=False):
    e = ToyTableEnv(num_envs=n, seed=seed, success_radius=radius)
    obs = e.reset()
    while not e.done.all():
        ids_, msk_ = tok.encode(obs["instruction"])
        chunk = model.sample_actions(
            torch.tensor(obs["image"], device=device), torch.tensor(ids_, device=device),
            torch.tensor(msk_, device=device), torch.tensor(obs["state"], device=device)).clamp(-1, 1).cpu().numpy()
        for k in range(exec_steps):
            obs, _, _, info = e.step(chunk[:, k])
    res = {"success": float(info["success"].mean()), "median_dist": float(np.median(info["dist"]))}
    return (res, e) if return_env else res

In [ ]:
# ✅ 自测
for r in [0.05, 0.03, 0.015]:
    print(f"成功半径 {r}: {evaluate(model, radius=r)}")
res = evaluate(model, radius=0.05)
assert res["success"] > 0.8
print("通过 ✅")

可视化几个回合（黑色圆环是夹爪的最终位置）：

In [ ]:
res, e = evaluate(model, n=8, radius=0.05, return_env=True)
fig, axes = plt.subplots(1, 8, figsize=(16, 2.4))
for i, ax in enumerate(axes):
    e.show(i, ax=ax)
    ax.set_title(f"{COLOR_EN[e.colors[i, e.target_idx[i]]]}: {'OK' if e.success[i] else 'fail'}", fontsize=9)
plt.show()

### 保存模型（第 8 章会用到）

In [ ]:
cache = default_cache_dir(); cache.mkdir(parents=True, exist_ok=True)
path = cache / f"mini_pi0_pi0_ep2000_h{H}.pt"
torch.save(model.state_dict(), path)
print("已保存基础 VLA 到", path, "——第 8 章的 vla_ladder.pipelines.get_base_vla() 会直接加载它")

---

## 5. 消融实验：没有 VLM 预训练会怎样？

完全相同的模型、相同的 2000 条示教、相同的训练步数，只是**跳过阶段 1**：

In [ ]:
set_seed(0)
model_scratch = MiniPi0(cfg).to(device)
hist_scratch = train_vla(model_scratch, data)
for r in [0.05, 0.03]:
    print(f"[无预训练] 成功半径 {r}: {evaluate(model_scratch, radius=r)}")
plt.figure(figsize=(5, 3)); plt.plot(np.convolve(hist_pre, np.ones(50) / 50, "valid"), label="with VLM pretraining")
plt.plot(np.convolve(hist_scratch, np.ones(50) / 50, "valid"), label="from scratch"); plt.yscale("log"); plt.legend(); plt.xlabel("step"); plt.ylabel("flow matching loss"); plt.show()

**解读**：从零训练的模型，训练 loss 也能降下来（它记住了训练集），但在新场景上成功率很低。原因是 2000 条示教只包含 2000 种方块布局，从这么少的数据中学会"看图 + 读指令 → 找到目标"太难了，模型更倾向于记忆。而预训练的 VLM 已经在 2 万张图上学会了定位，微调时只需要学"怎么走过去"。

> **这正是真实 VLA 的逻辑**：π0 在 ALOHA 上只用几十到几百条示教就能学会新任务，是因为 PaliGemma 已经在数十亿图文对上学会了"看"和"读"，`pi0_base` 又在 1 万小时机器人数据上学会了"动"。**所以微调时一定要从预训练权重出发，而且不要用太大的学习率把预训练知识冲掉**。

---

## 6. 读懂 loss 曲线：平台期意味着什么？

对比上面两条 loss 曲线：从零训练的模型，loss 会在 **0.35~0.4 附近停留几千步**才开始下降；有 VLM 预训练的模型几乎没有这个平台期。

原因：如果模型还不知道"目标方块在哪里"，对它来说最好的预测只能是"所有目标位置平均下来的速度"，这时的流匹配 loss 大约就是 0.35。从零训练时，模型要先花很多步学会"从图像和指令中找到目标"，才能突破这个平台；预训练过的 VLM 一开始就能提供目标位置信息，动作专家立刻就能利用它。

> **调试经验**：训练 VLA 时如果 loss 长时间卡在某个值不动，先算一下"忽略观测、只输出平均动作"时的 loss 是多少。如果两者接近，说明模型**还没有用上观测**（图像预处理错了、指令没送进去、相机名对不上……），而不是"需要更多训练"。

> 配套代码包也提供了 π0.5 变体（`MiniPi0Config(pi05=True)`：时间通过 adaRMSNorm 注入、状态离散化放进前缀）。它对学习率更敏感，作为课后探索题：试着把它训练到与 π0 变体相同的成功率，并记录你调整了哪些超参数。

---

## 小结

- 两阶段训练：VLM 预训练（视觉-语言对齐）→ VLA 训练（流匹配学动作块）。
- 动作块不能跨回合，padding 的动作不计入损失。
- 消融实验：没有 VLM 预训练，同样的示教数据下泛化差很多。这就是 VLA 必须基于预训练 VLM 的原因。
- loss 平台期 ≈ 模型还没有用上观测；预训练 VLM 让动作专家一开始就能获得目标信息。

🎉 你已经从零写出并训练了一个结构上与 π0 一致的 VLA！下一课去读真实的工业级代码库。